# NB02 scratch KWS (scratch arm)
Small CNN keyword classifier trained from scratch on Google Speech Commands v0.02 (35 classes, official split).
This is the "from scratch" arm of the scratch vs pretrained comparison. Closed vocabulary, cannot score phonemes.
Every number in the output JSON is measured by this run.

Settings: GPU T4, Internet ON (dataset downloads via torchaudio, about 2.3 GB).
If `torchaudio` fails to load audio, run `!pip install soundfile` or pin a torchaudio that matches the installed torch,
and note the fix in AI_USAGE_LOG.md.

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def device_info(force_cpu: bool = False) -> dict:
    name, threads = cpu_name(), os.cpu_count()
    try:
        import torch
        threads = torch.get_num_threads()
        if torch.cuda.is_available() and not force_cpu:
            name = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return {"name": name, "threads": threads, "platform": platform.platform()}


def versions(*pkgs: str) -> dict:
    out = {}
    for p in pkgs:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def write_result(notebook: str, data: dict, metrics: dict, notes: str, pkgs: tuple = ()) -> Path:
    """Shape from docs/NOTEBOOKS.md. Only values computed in this run go into metrics."""
    out = {"notebook": notebook, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "device": device_info(), "data": data, "metrics": metrics, "notes": notes,
           "versions": versions(*pkgs)}
    path = RESULTS / f"{notebook}.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out, indent=2, ensure_ascii=False))
    return path

In [ ]:
EPOCHS = 15
BATCH = 128
LR = 2e-3
SEED = 0
DATA_ROOT = WORK / "data"
DATA_ROOT.mkdir(exist_ok=True)

In [ ]:
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import DataLoader
from torchaudio.datasets import SPEECHCOMMANDS

SR = 16000
N_SAMPLES = SR  # 1 s clips
print("torch", torch.__version__, "| torchaudio", torchaudio.__version__, "| cuda", torch.cuda.is_available())


class Subset(SPEECHCOMMANDS):
    def __init__(self, split: str, root: str):
        super().__init__(root, download=True)

        def load(name: str) -> set[str]:
            with open(os.path.join(self._path, name)) as f:
                return {os.path.normpath(os.path.join(self._path, line.strip())) for line in f}

        walker = [os.path.normpath(w) for w in self._walker]
        if split == "validation":
            keep = load("validation_list.txt")   # read once, not once per file
            self._walker = [w for w in walker if w in keep]
        elif split == "testing":
            keep = load("testing_list.txt")
            self._walker = [w for w in walker if w in keep]
        else:
            excl = load("validation_list.txt") | load("testing_list.txt")
            self._walker = [w for w in walker if w not in excl]


def label_of(path: str) -> str:
    return Path(path).parent.name


def speaker_of(path: str) -> str:
    return Path(path).name.split("_nohash_")[0]


train, val, test = (Subset(s, str(DATA_ROOT)) for s in ("training", "validation", "testing"))
labels = sorted({label_of(w) for w in train._walker})
print(f"train {len(train)} | val {len(val)} | test {len(test)} | labels {len(labels)}")

In [ ]:
mel = torchaudio.transforms.MelSpectrogram(SR, n_fft=400, hop_length=160, n_mels=40)
label_idx = {l: i for i, l in enumerate(labels)}


def collate(batch):
    xs, ys = [], []
    for wav, _sr, label, *_ in batch:
        wav = wav[:, :N_SAMPLES]
        if wav.shape[1] < N_SAMPLES:
            wav = nn.functional.pad(wav, (0, N_SAMPLES - wav.shape[1]))
        xs.append(torch.log(mel(wav) + 1e-6))
        ys.append(label_idx[label])
    return torch.stack(xs), torch.tensor(ys)


class SmallCNN(nn.Module):
    def __init__(self, n_classes: int):
        super().__init__()

        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

        self.net = nn.Sequential(block(1, 16), block(16, 32), block(32, 64), nn.AdaptiveAvgPool2d(1),
                                 nn.Flatten(), nn.Dropout(0.2), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def evaluate(model, loader, dev) -> float:
    model.eval()
    ok = total = 0
    for x, y in loader:
        ok += (model(x.to(dev)).argmax(1).cpu() == y).sum().item()
        total += len(y)
    return ok / total

In [ ]:
torch.manual_seed(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tl = DataLoader(train, BATCH, shuffle=True, collate_fn=collate, num_workers=2)
vl = DataLoader(val, BATCH, collate_fn=collate, num_workers=2)
te = DataLoader(test, BATCH, collate_fn=collate, num_workers=2)

model = SmallCNN(len(labels)).to(dev)
opt = torch.optim.AdamW(model.parameters(), LR)
loss_fn = nn.CrossEntropyLoss()
best, best_state, history, t0 = -1.0, None, [], time.time()
for ep in range(EPOCHS):
    model.train()
    te0 = time.time()
    for x, y in tl:
        opt.zero_grad()
        loss_fn(model(x.to(dev)), y.to(dev)).backward()
        opt.step()
    v = evaluate(model, vl, dev)
    history.append({"epoch": ep + 1, "val_acc": v, "epoch_seconds": time.time() - te0})
    print(f"epoch {ep + 1}/{EPOCHS} val_acc {v:.4f} ({history[-1]['epoch_seconds']:.0f} s)", flush=True)
    if v > best:
        best, best_state = v, {k: t.clone() for k, t in model.state_dict().items()}
model.load_state_dict(best_state)
train_s = time.time() - t0
test_acc = evaluate(model, te, dev)
torch.save(best_state, WORK / "kws_scratch.pt")
print("test_acc", test_acc)

In [ ]:
# CPU latency for one 1 s clip, 1 thread, median of 100 runs after 10 warm-up runs. Model only, mel excluded.
cpu_model = SmallCNN(len(labels))
cpu_model.load_state_dict({k: v.cpu() for k, v in best_state.items()})
cpu_model.eval()
x = torch.log(mel(torch.randn(1, N_SAMPLES)) + 1e-6).unsqueeze(0)
torch.set_num_threads(1)
times = []
with torch.inference_mode():
    for i in range(110):
        t = time.perf_counter()
        cpu_model(x)
        if i >= 10:
            times.append(time.perf_counter() - t)
times.sort()

In [ ]:
all_paths = train._walker + val._walker + test._walker
write_result(
    "NB02_scratch_kws",
    data={"source": "Google Speech Commands v0.02 (official split)", "n_utterances": len(all_paths),
          "speakers": len({speaker_of(p) for p in all_paths}),
          "n_train": len(train), "n_val": len(val), "n_test": len(test), "labels": len(labels)},
    metrics={"best_val_acc": best, "test_acc": test_acc, "train_seconds": round(train_s, 1),
             "epochs": EPOCHS, "params": sum(p.numel() for p in cpu_model.parameters()),
             "cpu_latency_ms_median_1s_clip_1thread": times[len(times) // 2] * 1000,
             "cpu_latency_cpu": cpu_name(), "history": history},
    notes=f"SmallCNN, log-mel 40, AdamW lr={LR}, batch={BATCH}, seed={SEED}. Best epoch by val acc. "
          "CPU latency measured on the Kaggle CPU, not the laptop.",
    pkgs=("torch", "torchaudio"),
)